pipeline functions
- most bassic / high level api of transformer library
- puts together all steps from raw text to prediction
- includes all prprocessing and post proessing

In [9]:
#eg sentiment analysis
from transformers import pipeline
classifier = pipeline("sentiment-analysis",device="cuda")
classifier("I hate you but i still love you")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9996654987335205}]

In [13]:
summarization = pipeline("summarization",device="cuda",max_length=20)
summarization("The year was 1944. As Allied and Axis forces clashed across Europe, an event unfolded that no history book would ever record. Without warning, strange emerald lights appeared over the battlefields, descending silently through the clouds. From them emerged towering alien machines unlike anything humanity had imagined.")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Your min_length=56 must be inferior than your max_length=20.
c:\Users\Admin\anaconda3\lib\site-packages\transformers\generation\utils.py:1282: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (20). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


[{'summary_text': ' As Allied and Axis forces clashed across Europe, an event unfolded that no history book would'}]

zero shot classification tasks
- allows users to input labels that models are not explicitly trained on

models
- can use models in hugging face by passing in the model argument, check website for specific models



Transformers
- most trans models are langugage models, trained on large amt of raw texts
- in a self supervised manner meaning humans not needed to compute labels, do tasks on unlabeled data
- they are general models, need fine tuning to perform btre on specifci tasks
- casual language modeling, predict next token based on cur and prev tokens

Transfer learning
- teacher model, trained on large amt of data
- for a specifci trask, use model B ut use ther weights initialised fromn model a weights
- as initilasing with random weights is easier and simpler
- fine tuning with pretrained model
- fine tuning is a type of transfer learning

Transformer architecture
- encoder + decoder
- can be used tgt and independently
- encoder encodes data into output embeddings (data like text img into numerical representations)
- uses self attention
- decoder uses masked self at but differs from encoder as it is uni directional and autoregrssive
- enocder encodes highh level representation and pass to decoder and deocder uses input in auto regressivemanner to preedfict ouputs

Q, K, V - same or different across tokens/encoder/decoder?

1) within one sequence - same weights, different output per token
- every token uses the SAME W_q, W_k, W_v (shared weight matrices)
- but each token has a different input x, so each token still gets its OWN q, k, v vectors
- eg: token "cat" and token "sat" both go through the same W_q, but produce different q vectors since their inputs differ

2) across encoder vs decoder - completely separate weights, not shared
- encoder self-attn, decoder masked self-attn, and decoder cross-attn each have
  their OWN independently trained W_q, W_k, W_v - none of them are reused

| attention layer          | Q comes from      | K,V come from        | job                                  |
|---------------------------|--------------------|------------------------|----------------------------------------|
| encoder self-attn         | encoder tokens     | encoder tokens         | understand full input (bidirectional)  |
| decoder masked self-attn  | decoder tokens     | decoder tokens (past)  | understand what's been generated so far|
| decoder cross-attn        | decoder tokens     | encoder OUTPUT         | look up relevant info from source text |

- even cross-attn's K,V (applied to encoder output) use their OWN separate weights,
  not the encoder's own internal W_k/W_v

simple takeaway: same q/k/v RECIPE everywhere, but 3 separate sets of learned
weights (encoder self-attn / decoder self-attn / decoder cross-attn), each
producing per-token vectors from shared-within-that-layer weights

Decoder attention order: masked self-attn FIRST, then cross-attn, then feedforward
(this whole 3-step sequence = ONE decoder layer, repeated n_layer times)

1) masked self-attn first
- decoder looks at its OWN past output tokens (not the encoder) to build an
  understanding of "what have I said so far / what's the meaning of my own
  output up to this point" - causal, can't see its own future tokens

2) cross-attn second
- NOW, using that refined self-understanding as the query, it looks over at
  the encoder's output (K,V) to ask "given where I am, what parts of the
  SOURCE sentence are relevant right now"
- query comes from step 1's result, not the raw input - so cross-attn is
  informed by what the decoder already figured out about itself first

3) feedforward last
- combines/processes what was gathered from both self-attn (self) and
  cross-attn (source sentence) per token

important: none of these 3 steps is "the moment a token gets generated" -
masked self-attn/cross-attn/feedforward just build a refined per-token
representation. actual next-token output only happens AFTER all n_layer
decoder layers finish, when the final layer's last-position output gets
projected through one final linear layer -> softmax -> sampled

"autoregressive" = the OUTER LOOP: run the ENTIRE decoder stack (all layers,
all 3 sublayers) on the current sequence -> take last position's logits ->
sample next token -> append it -> re-run the WHOLE stack again from scratch
on the now-longer sequence. masked self-attn happens fresh on EVERY loop
iteration, not just once.

How language models work
- learn statistical rls btw words
- predict next token
- diff arch types encoder , decoder , both
- encoder only good for understanding text
- deoder only goof for generating
- both combined
- eg gpt2 decoder only good at creating text
- text classification -> encoder only good
- token classficication -> assign labels

2 main approches for training
- masked -> randomly mask tokens left and right -> model learns bi directional context
- casual -> mask all future tokens so only use context from past to predict